In [ ]:
# αν θες να δουλεύεις τοπικά, αγνόησε το κελί αυτό

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
base_dir = "/content/drive/SharedDrives/YOLO26"
# Αν δουλεύεις σε google drive,
# αντικατέστησέ το SharedDrives --> *MyDrive* (εγώ δούλευα σε Shared Drive folder)
# αν θες να δουλεύεις τοπικά, αγνόησε το κελί αυτό


# YOLO Notebook


## 1. Install / import dependencies

In [ ]:
pip install ultralytics moviepy opencv-python torch

In [ ]:
from pathlib import Path
import os
import subprocess
import cv2
import torch
import moviepy.config as moviepy_config
from ultralytics import YOLO
import albumentations as A
import numpy as np
from tqdm import tqdm

print("Imports loaded successfully.")

Imports loaded successfully.


## 2. Main paths and settings

Change these paths depending on your project folder.


In [ ]:
# -----------------------------
# MOV to MP4 conversion settings
# -----------------------------

# Γεωργία: Στην περιπτωση που δουλεύεις τοπικά:
# Εδώ ορίζουμε τον φάκελο του project, οπότε αλλάζεις το path ανάλογα με το που έχεις αποθηκεύσει τα αρχεία σου.
# Προκειμένου να μην χρειαστεί να κάνεις αλλαγές στον κώδικα, καλο θα είναι να έχεις μέσα στον φάκελο,
# υποφακέλους με τα ονόματα "training_dataset", "models", "prediction", και μέσα στον φάκελο "prediction"
# να δημιουργήσεις υποφάκελο με τα ονόμα "images_test".

# Στην περίπτωση που δουλεύεις σε google collab notebook
# αντικατέστησέ μονο το SharedDrives --> *MyDrive*

VIDEO_FOLDER = Path(r"/content/drive/Shareddrives/YOLO26/proofreading/videos")


# -----------------------------
# YOLO training settings
# -----------------------------

BASE_MODEL = "yolo26n.pt"
DATA_YAML = Path(r"/content/drive/Shareddrives/YOLO26/proofreading/training_dataset/data.yaml")

EPOCHS = 150
IMAGE_SIZE = 512

# Training-performance settings. These are used for BOTH Drive and local-ZIP training.
DEVICE = 0                 # First CUDA GPU in Colab.
WORKERS = 2                # Safe default for Drive; local-ZIP setup raises this when possible.
BATCH_SIZE = 0.70          # Auto-select batch size targeting about 70% GPU memory. Use 32 for a fixed batch.
USE_AMP = True             # Automatic mixed precision for faster/lighter GPU training.
CACHE_IMAGES = True        # Cache images in RAM after the first read.
RECT_TRAINING = True       # Rectangular batches; change to False for standard shuffled square batches.


# -----------------------------
# YOLO prediction settings
# -----------------------------

MODEL_PATH = Path(r"/content/drive/Shareddrives/YOLO26/proofreading/models/YOLO26_test_1/weights/best.pt")
PREDICTION_SOURCE = Path(r"/content/drive/Shareddrives/YOLO26/proofreading/prediction/images_test")

PREDICT_PROJECT = Path(r"/content/drive/Shareddrives/YOLO26/proofreading/prediction")
PREDICT_NAME = "results"

CONFIDENCE = 0.35
LINE_WIDTH = 2

print("Settings are ready.")

Settings are ready.


## 3. Convert `.MOV` videos to `.mp4` and extract the videos' frames

- use FFmpeg
- convert to H.264
- use `yuv420p` for compatibility
- use `crf=18` for high quality
- remove audio with `-an`


In [ ]:
# Η μετατροπή γίνεται με την βιβλιοθήκη moviepy, η οποία χρησιμοποιεί το ffmpeg.
# Αν δεν έχει εγκαταστήσει το ffmpeg, μπορείς να το κατεβάσεις από εδώ: https://ffmpeg.org/download.html
# Η μετατροπή αυτή είναι απαραίτητη γιατί το YOLO δεν υποστηρίζει αρχεία .MOV (τύπος αρχείου βίντεο του iphone), αλλά μόνο .MP4.

def convert_mov_to_mp4(input_path, output_path):
    """Convert one MOV file to MP4 using FFmpeg."""

    ffmpeg_path = moviepy_config.FFMPEG_BINARY

    command = [
        ffmpeg_path,
        "-y",
        "-i", str(input_path),
        "-vcodec", "libx264",
        "-pix_fmt", "yuv420p",
        "-crf", "18",
        "-an",
        str(output_path),
    ]

    try:
        subprocess.run(
            command,
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            check=True,
        )
        return True

    except subprocess.CalledProcessError as error:
        print(f"FFmpeg failed for: {input_path.name}")
        print(error.stderr.decode(errors="ignore"))
        return False

    except Exception as error:
        print(f"Unexpected error for {input_path.name}: {error}")
        return False


def convert_folder_mov_to_mp4(video_folder):
    """Convert all .MOV files inside a folder."""

    video_folder = Path(video_folder)

    if not video_folder.exists():
        print(f"Folder does not exist: {video_folder}")
        return

    mov_files = sorted(video_folder.glob("*.MOV")) + sorted(video_folder.glob("*.mov"))

    if not mov_files:
        print(f"No .MOV files found in: {video_folder}")
        return

    print(f"Found {len(mov_files)} MOV files.")

    for index, input_file in enumerate(mov_files, start=1):
        output_file = input_file.with_suffix(".mp4")

        print(f"[{index}/{len(mov_files)}] Converting {input_file.name} -> {output_file.name} ...", end=" ")

        success = convert_mov_to_mp4(input_file, output_file)

        if success:
            print("done")
        else:
            print("failed")

    print("Conversion finished.")

In [ ]:
# Run this cell only when you want to convert videos

convert_folder_mov_to_mp4(VIDEO_FOLDER)

In [ ]:
def extract_frames(video_path, output_folder, frame_skip=10):
    """
    Extracts frames from a video file.
    :param video_path: Path to the input video.
    :param output_folder: Where to save the images.
    :param frame_skip: Saves every Nth frame (e.g., 10 means 1 frame saved every 10 frames).
    """
    print(f"Opening video: {video_path}")

    video = cv2.VideoCapture(video_path)

    if not video.isOpened():
        print("Error: Could not open the video. Check the file path.")
        return

    os.makedirs(output_folder, exist_ok=True)

    frame_count = 0
    saved_count = 0

    print("Extracting frames... ")

    while True:
        success, frame = video.read()

        if not success:
            break

        # Only save every Nth frame
        if frame_count % frame_skip == 0:
            # Create a file name with leading zeros (e.g., frame_0001.jpg)
            file_name = f"frame_{saved_count:04d}.jpg"
            save_path = os.path.join(output_folder, file_name)

            # Save the image
            cv2.imwrite(save_path, frame)
            saved_count += 1

        frame_count += 1

    # Cleanup
    video.release()
    print(f"\nDone! Extracted {saved_count} frames to {output_folder}")


In [ ]:
# Run this cell only when you want to extract frames.
# To do so, change the name of the mp4. Make sure your video is in the correct folder as stated in variable VIDEO_FOLDER

extract_frames(VIDEO_FOLDER / "IMG_0018.mp4", PREDICTION_SOURCE, frame_skip=10)

## 4. Train YOLO


Note για Γεωργία. 771 είναι τα frames όπως προέκυψαν από τα videos. Στα 771 frames σημειώθηκαν 3 labels (όπου υπήρχαν αντίστοιχα) :
1) traffic_signs (3005)
2) traffic_lights (1590)
3) auth_label (538)

Για το σκοπό αυτό χρησημοποιήσαμε το ROBOFLOW, μία πλατφόρμα που απλοποιεί
1) Επισήμανση (Labeling): Διαθέτει εύχρηστο περιβάλλον ιστού για να σχεδιάζονται πλαίσια (bounding boxes), να ταξινομούνται ή να απομονώνονται αντικείμενα (χειροκίνητη σήμανση)
2) Προεπεξεργασία & Αυξημένη επεξεργασία (Augmentation): Μπορεί αυτόματα να πολλαπλασιάσει τα δεδομένα.

Για το augmentation επιλέχθηκε στο roboflow
1) 90° Rotate: Clockwise, Counter-Clockwise
2) Grayscale: Apply to 15% of images
3) Hue: Between -15° and +15°

Φτάνοντας και εξάγοντας dataset με σύνολο 1849 labeled εικόνες χωρισμένες με λογική 70-20-10 (training-validation-test). Το dataset εξάγεται με τους αντίστοιχους φακέλους train-valid-test (θα τους βρεις στον φάκελο training_dataset), και το yaml αρχείο που ορίζει τον κάθε φάκελο για το yolo26n.

Στη συνέχεια κάναμε επιπλέον augmentation μέσω του yolo, προσεγγίζοντας τα 6000 frames.
1) Αλλαγή φωτεινότητας/αντίθεσης (συννεφιά/ήλιος)
2) Μικρή στροφή/κούνημα
3) Προσομοίωση κίνησης αυτοκινήτου (motion blur)

In [ ]:
import os
import cv2
import albumentations as A
import numpy as np
from tqdm import tqdm

# Input & output directories
images_dir = "/content/yolo_dataset/CH_test_2/train/images"
labels_dir = "/content/yolo_dataset/CH_test_2/train/labels"

output_images_dir = "/content/yolo_dataset/CH_test_3/train/images"
output_labels_dir = "/content/yolo_dataset/CH_test_3/train/labels"


os.makedirs(output_images_dir, exist_ok=True)
os.makedirs(output_labels_dir, exist_ok=True)

def read_labels(label_path):
    if not os.path.exists(label_path): return []
    with open(label_path, 'r') as f:
        return [line.strip() for line in f.readlines() if line.strip()]

def save_labels(output_path, lines):
    with open(output_path, 'w') as f:
        f.write('\n'.join(lines) + '\n')

image_files = [f for f in os.listdir(images_dir) if f.lower().endswith(('.jpg', '.png', '.jpeg'))]
print(f"Found {len(image_files)} segmentation images. Starting the correct conversion...")

for img_file in tqdm(image_files):
    base_name = os.path.splitext(img_file)[0]
    img_path = os.path.join(images_dir, img_file)
    label_path = os.path.join(labels_dir, base_name + ".txt")

    img = cv2.imread(img_path)
    if img is None: continue
    labels = read_labels(label_path)

    if not labels: continue

    # Brightness
    cv2.imwrite(os.path.join(output_images_dir, img_file), img)
    save_labels(os.path.join(output_labels_dir, base_name + ".txt"), labels)

    img_bright = cv2.convertScaleAbs(img, alpha=1.2, beta=30)
    cv2.imwrite(os.path.join(output_images_dir, base_name + "_bright.jpg"), img_bright)
    save_labels(os.path.join(output_labels_dir, base_name + "_bright.txt"), labels)

    # Flip - Rotation
    flipped_labels = []
    for line in labels:
        parts = line.split()
        class_id = parts[0]
        coords = parts[1:]

        new_coords = []
        for i in range(0, len(coords), 2):
            x = float(coords[i])
            y = float(coords[i+1])

            new_x = 1.0 - x
            new_coords.append(f"{new_x:.6f}")
            new_coords.append(f"{y:.6f}")

        flipped_labels.append(f"{class_id} " + " ".join(new_coords))

    # Save
    img_flip = cv2.flip(img, 1)
    cv2.imwrite(os.path.join(output_images_dir, base_name + "_flip.jpg"), img_flip)
    save_labels(os.path.join(output_labels_dir, base_name + "_flip.txt"), flipped_labels)

    img_flip_bright = cv2.convertScaleAbs(img_flip, alpha=1.2, beta=30)
    cv2.imwrite(os.path.join(output_images_dir, base_name + "_flip_bright.jpg"), img_flip_bright)
    save_labels(os.path.join(output_labels_dir, base_name + "_flip_bright.txt"), flipped_labels)


Iterationg through 1618 images...


  0%|          | 3/1618 [00:04<44:11,  1.64s/it]


KeyboardInterrupt: 

OLD, DO NOT RUN, GO TO THE ALTERNATIVE BELOW

In [ ]:
def train_yolo():
    """Train YOLO with the settings defined above."""

    if not DATA_YAML.exists():
        print(f"data.yaml not found: {DATA_YAML}")
        return None

    model = YOLO(BASE_MODEL)
    results = model.train(
        data=str(DATA_YAML),
        epochs=EPOCHS,
        imgsz=IMAGE_SIZE,
        # device = 0,
        workers=2,
        batch=32,
        amp=True,
        cache=True,
        rect=True,
        # device=DEVICE, #comment if you don't have CUDA available and want to use GPU. In this case uncomment next row
        device='cpu',
        # workers=WORKERS,
        project=str(TRAIN_PROJECT),
        name=TRAIN_NAME,
        exist_ok=True,
    )

    return results

### Alternative setup: train from a ZIP extracted into Colab's local `/content` storage

Use this setup instead of reading thousands of individual image and label files from mounted Google Drive. Training reads the extracted local files much faster.

**How to add a ZIP to `/content`:**

1. Open Colab's **Files** panel using the folder icon on the left.
2. Click **Upload to session storage** and select the `.zip` file. It will appear at `/content/your_file.zip`.
3. In the cell below, set `UPLOAD_WITH_WIDGET = False` and update `ZIP_PATH` with that filename.

Alternatively, leave `UPLOAD_WITH_WIDGET = True`. Running the cell opens a browser file picker and uploads the selected ZIP directly into `/content`.

Run this setup cell **before** the normal training cell below. Skip it when you want to use the original Google Drive paths.

> `/content` is temporary. The uploaded dataset and locally saved runs disappear when the Colab runtime resets, so copy or download the final weights afterward.


In [ ]:
def train_yolo():
    """Train YOLO using the active dataset paths and named training settings."""

    # if not DATA_YAML.exists():
    #     print(f"data.yaml not found: {DATA_YAML}")
    #     return None

    # if DEVICE == 0 and not torch.cuda.is_available():
    #     raise RuntimeError(
    #         "DEVICE=0 requests a GPU, but CUDA is unavailable. "
    #         "In Colab choose Runtime > Change runtime type > GPU, then run the notebook again."
    #     )

    BASE_MODEL = "yolo26n.pt"
    EPOCHS = 150
    IMAGE_SIZE = 512
    DEVICE = 0
    WORKERS = 2
    BATCH_SIZE = 0.7
    USE_AMP = True
    CACHE_IMAGES = True
    RECT_TRAINING = True

    effective_settings = {
        "model": BASE_MODEL,
        "data": str(DATA_YAML),
        "epochs": EPOCHS,
        "imgsz": IMAGE_SIZE,
        "device": DEVICE,
        "workers": WORKERS,
        "batch": BATCH_SIZE,
        "amp": USE_AMP,
        "cache": CACHE_IMAGES,
        "rect": RECT_TRAINING,
        "project": str(TRAIN_PROJECT),
        "name": TRAIN_NAME,
    }

    print("Effective training configuration:")
    for key, value in effective_settings.items():
        print(f"  {key}: {value}")

    model = YOLO(BASE_MODEL)
    results = model.train(
        data=str(DATA_YAML),
        epochs=EPOCHS,
        imgsz=IMAGE_SIZE,
        device= DEVICE,
        workers=WORKERS,
        batch=BATCH_SIZE,
        amp=USE_AMP,
        cache=CACHE_IMAGES,
        rect=RECT_TRAINING,
        project=str(TRAIN_PROJECT),
        name=TRAIN_NAME,
        exist_ok=True,
        patience=15,
    )

    return results

In [ ]:
# Upload ZIP -> extract under /content -> point the existing training function to local files.
from google.colab import files
from pathlib import Path, PurePosixPath
import os
import re
import shutil
import subprocess

# OPTION 1: True opens a file picker and uploads the ZIP directly to /content.
# OPTION 2: False uses a ZIP already uploaded through Colab's Files panel.
UPLOAD_WITH_WIDGET = False
ZIP_PATH = Path("/content/CH_test_3.zip")  # Edit this for Option 2.
EXTRACT_DIR = Path("/content/yolo_dataset")
LOCAL_TRAIN_PROJECT = Path("/content/yolo_runs")

if UPLOAD_WITH_WIDGET:
    os.chdir("/content")
    uploaded = files.upload()
    zip_names = [name for name in uploaded if name.lower().endswith(".zip")]
    if not zip_names:
        raise ValueError("No .zip file was uploaded. Run the cell again and select a ZIP file.")

    # Explicitly write and close the uploaded bytes before extraction. This avoids
    # file-handle/seek errors that can occur with Python's zipfile.extractall().
    selected_name = zip_names[0]
    ZIP_PATH = Path("/content") / Path(selected_name).name
    with open(ZIP_PATH, "wb") as f:
        f.write(uploaded[selected_name])
        f.flush()
        os.fsync(f.fileno())
    del uploaded

if not ZIP_PATH.exists():
    raise FileNotFoundError(
        f"ZIP not found: {ZIP_PATH}. Upload it in Colab's Files panel or use the upload widget."
    )
if ZIP_PATH.stat().st_size == 0:
    raise ValueError(f"The uploaded ZIP is empty: {ZIP_PATH}")

print(f"ZIP size: {ZIP_PATH.stat().st_size / (1024**2):,.1f} MB")

# Validate the archive with the operating-system unzip utility. It is generally
# more reliable than Python's zipfile module for large datasets in Colab.
validation = subprocess.run(
    ["unzip", "-tqq", str(ZIP_PATH)],
    text=True,
    capture_output=True,
)
if validation.returncode != 0:
    details = (validation.stderr or validation.stdout).strip()
    raise RuntimeError(
        "ZIP validation failed. The upload may be incomplete or the archive may be damaged. "
        "Re-create/re-upload the ZIP, then try again.\n" + details[-2000:]
    )

# Reject unsafe paths before extracting.
listing = subprocess.run(
    ["unzip", "-Z1", str(ZIP_PATH)],
    text=True,
    capture_output=True,
    check=True,
)
for raw_name in listing.stdout.splitlines():
    normalized = raw_name.replace(chr(92), "/")
    parts = PurePosixPath(normalized).parts
    if (
        normalized.startswith("/")
        or normalized.startswith(chr(92))
        or ".." in parts
        or (parts and re.fullmatch(r"[A-Za-z]:", parts[0]))
    ):
        raise ValueError(f"Unsafe ZIP entry detected: {raw_name}")

# Start with a clean extraction folder.
if EXTRACT_DIR.exists():
    shutil.rmtree(EXTRACT_DIR)
EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

# Extract with the system utility rather than zipfile.extractall().
extraction = subprocess.run(
    ["unzip", "-q", "-o", str(ZIP_PATH), "-d", str(EXTRACT_DIR)],
    text=True,
    capture_output=True,
)
if extraction.returncode != 0:
    details = (extraction.stderr or extraction.stdout).strip()
    raise RuntimeError("Extraction failed:\n" + details[-2000:])

# Locate data.yaml/data.yml even if the ZIP contains an extra top-level folder.
yaml_candidates = list(EXTRACT_DIR.rglob("data.yaml")) + list(EXTRACT_DIR.rglob("data.yml"))
yaml_candidates = sorted(set(yaml_candidates), key=lambda p: (len(p.parts), str(p)))
if not yaml_candidates:
    raise FileNotFoundError(
        f"No data.yaml or data.yml was found inside {EXTRACT_DIR}. Check the ZIP's folder structure."
    )

if len(yaml_candidates) > 1:
    print("Multiple dataset YAML files found. Using the first one:")
    for candidate in yaml_candidates:
        print(" -", candidate)

# Override the original Drive-based variables used by train_yolo().
DATA_YAML = yaml_candidates[0]
TRAIN_PROJECT = LOCAL_TRAIN_PROJECT
TRAIN_NAME = "YOLO26_local"

print(f"ZIP:          {ZIP_PATH}")
print(f"Extracted to: {EXTRACT_DIR}")
print(f"data.yaml:    {DATA_YAML}")
print(f"Runs saved:   {TRAIN_PROJECT / TRAIN_NAME}")
print("Local setup is ready. Run the next training cell.")


ZIP size: 364.8 MB
ZIP:          /content/CH_test_3.zip
Extracted to: /content/yolo_dataset
data.yaml:    /content/yolo_dataset/CH_test_3/data.yaml
Runs saved:   /content/yolo_runs/YOLO26_local
Local setup is ready. Run the next training cell.


In [ ]:
# Run this cell only when you want to train the model

training_results = train_yolo()

Effective training configuration:
  model: yolo26n.pt
  data: /content/yolo_dataset/CH_test_3/data.yaml
  epochs: 150
  imgsz: 512
  device: 0
  workers: 2
  batch: 0.8
  amp: True
  cache: True
  rect: True
  project: /content/yolo_runs
  name: YOLO26_local
Ultralytics 8.4.90 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=0.8, bgr=0.0, box=7.5, cache=True, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/yolo_dataset/CH_test_3/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=150, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=512, iou=0.7, keras=F

KeyboardInterrupt: 

## 5. Run YOLO prediction


In [ ]:
def predict_yolo():
    """Run prediction on a folder of images."""

    if not MODEL_PATH.exists():
        print(f"Model not found: {MODEL_PATH}")
        return None

    if not PREDICTION_SOURCE.exists():
        print(f"Prediction folder not found: {PREDICTION_SOURCE}")
        return None

    model = YOLO(str(MODEL_PATH))

    results = model.predict(
        source=str(PREDICTION_SOURCE),
        save=True,
        imgsz=IMAGE_SIZE,
        conf=CONFIDENCE,
        line_width=LINE_WIDTH,
        show_labels=False,
        project=str(PREDICT_PROJECT),
        name=PREDICT_NAME,
        exist_ok=True,
    )

    return results

In [ ]:
# Run this cell only when you want to predict

prediction_results = predict_yolo()